# 8주차 실습 — SQL (2) 집계와 기본 JOIN

**데이터베이스 관리 · 8주차 실습**

- 오늘 배우는 것: 집계 함수(`COUNT` · `SUM` · `AVG` · `MIN` · `MAX` · `MEDIAN`) · `GROUP BY` · `HAVING` · 작성 순서와 실행 순서 · `JOIN`(INNER, LEFT) · JOIN 뒤 행 수 확인
- 지난 시간: `SELECT → FROM → WHERE → ORDER BY → LIMIT`. 오늘은 그 사이에 `GROUP BY → HAVING` 이 들어갑니다.

## 0. 준비
아래 셀을 실행하면 데이터가 연결됩니다. 오류가 나면 런타임을 다시 시작한 뒤 한 번 더 실행하세요.

In [ ]:
# ── 0. 준비: 이 셀은 그대로 실행만 하세요 (Shift+Enter) ─────────────────────────
import os, sys, urllib.request
DB  = "aact_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini.duckdb.gz"

# 로컬(수업 폴더)에서 열었을 때는 ../data/ 의 파일을 사용
for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand
        break
else:
    print("데이터 파일을 내려받습니다 (약 50MB, 1분 내외) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")
    print("완료:", DB)

if "google.colab" in sys.modules:           # Colab 에서만 설치
    import subprocess; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "jupysql", "duckdb-engine"])

import duckdb, pandas as pd, hashlib
pd.set_option("display.max_colwidth", 60)
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
conn = duckdb.connect(DB, read_only=True)
%sql conn --alias aact
print("연결 완료:", DB)

def _sig(df):
    """결과표의 내용 지문: 열 이름 소문자화, 소수 4자리 반올림, 행 정렬 후 해시"""
    d = df.copy()
    d.columns = [str(c).lower() for c in d.columns]
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]):
            d[c] = d[c].round(4)
    d = d.astype(str).sort_values(by=list(d.columns)).reset_index(drop=True)
    return hashlib.md5(d.to_csv(index=False).encode()).hexdigest()[:12]

def check(name, df):
    """직전 %%sql 결과를 _ 로 넘겨 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print("알 수 없는 문항:", name); return
    if not isinstance(df, pd.DataFrame):
        print("❌ 결과가 표가 아닙니다. %%sql 셀을 먼저 실행한 뒤, 바로 다음 셀에서 check('%s', _) 를 실행하세요." % name); return
    exp = ANSWERS[name]
    if _sig(df) == exp["sig"]:
        print(f"✅ {name} 정답!  (행 {len(df)}개)")
    else:
        hint = exp.get("hint", "")
        rows = exp.get("rows")
        msg = f"❌ {name} 아직 다릅니다."
        if rows is not None:
            msg += f"  기대 행 수 {rows}개, 현재 {len(df)}개."
        if hint:
            msg += "  힌트: " + hint
        print(msg)

In [ ]:
ANSWERS = {
 "O1": {
  "sig": "52c16eece2e5",
  "hint": "SELECT → FROM → WHERE → GROUP BY → ORDER BY → LIMIT",
  "rows": 5
 },
 "O2": {
  "sig": "da9ba83ad733",
  "hint": "HAVING 은 GROUP BY 뒤, ORDER BY 앞",
  "rows": 6
 },
 "Q1": {
  "sig": "499962c8fdb3",
  "hint": "GROUP BY study_type ORDER BY n DESC",
  "rows": 2
 },
 "Q2": {
  "sig": "560b9766a944",
  "hint": "두 조건을 AND 로",
  "rows": 1
 },
 "Q3": {
  "sig": "b15b2b8d838e",
  "hint": "WHERE phase = 'PHASE3', GROUP BY 시작연도",
  "rows": 10
 },
 "Q4": {
  "sig": "10ca15a3637b",
  "hint": "HAVING COUNT(*) >= 1500",
  "rows": 6
 },
 "Q5": {
  "sig": "913571f55842",
  "hint": "WHERE removed = false, HAVING COUNT(DISTINCT nct_id) >= 2000",
  "rows": 3
 },
 "Q6": {
  "sig": "335613f25abb",
  "hint": "ON s.nct_id = c.nct_id, 'South Korea', 'PHASE3'",
  "rows": 1
 },
 "Q7": {
  "sig": "e1a03621263e",
  "hint": "ON s.nct_id = i.nct_id, GROUP BY i.intervention_type",
  "rows": 11
 },
 "Q8": {
  "sig": "fbb66c51f343",
  "hint": "'Heart Failure', 'lead'",
  "rows": 7
 }
}

## 1. 집계 함수 — 여러 행을 하나의 숫자로

| 함수 | 뜻 | 주의 |
|---|---|---|
| `COUNT(*)` | 행 수 | NULL 포함해서 셈 |
| `COUNT(열)` | 그 열에 **값이 있는** 행 수 | NULL 은 세지 않음 |
| `COUNT(DISTINCT 열)` | 서로 다른 값의 수 | 중복 제거 |
| `SUM` · `AVG` · `MIN` · `MAX` | 합계 · 평균 · 최소 · 최대 | NULL 은 무시 |
| `MEDIAN` | 중앙값 | 극단값에 덜 흔들림 |

In [ ]:
%%sql
-- COUNT 세 가지 비교
SELECT COUNT(*) AS 전체행수,
       COUNT(results_first_posted_date) AS 결과게시일있음,
       COUNT(DISTINCT overall_status) AS 상태종류
FROM studies

In [ ]:
%%sql
-- 등록 인원의 요약 통계 (중재 연구만)
SELECT MIN(enrollment) AS 최소, MAX(enrollment) AS 최대,
       ROUND(AVG(enrollment)) AS 평균, MEDIAN(enrollment) AS 중앙값
FROM studies
WHERE study_type = 'INTERVENTIONAL'

> 평균이 중앙값보다 훨씬 큽니다. 지난 시간에 본 1억 명짜리 입력 오류 같은 **극단값이 평균을 끌어올리기** 때문입니다.
> 병원 통계에서도 재원일수·진료비처럼 한쪽으로 치우친 값은 **평균과 중앙값을 함께** 보고, 필요하면 극단값을 확인한 뒤 제외합니다.

In [ ]:
%%sql
-- 등록 인원 1만 명 이하로 제한하면 평균이 어떻게 바뀌는지
SELECT ROUND(AVG(enrollment)) AS 평균, MEDIAN(enrollment) AS 중앙값, COUNT(*) AS n
FROM studies
WHERE study_type = 'INTERVENTIONAL' AND enrollment <= 10000

In [ ]:
%%sql
-- 비율 계산: 완료된 시험 중 결과가 게시된 비율(%)  → 100.0 을 곱해 소수로 계산
SELECT ROUND(100.0 * COUNT(results_first_posted_date) / COUNT(*), 1) AS 결과게시율_퍼센트
FROM studies
WHERE overall_status = 'COMPLETED'

## 2. GROUP BY — 묶어서 세기

```
SELECT   묶는열, COUNT(*) AS n        -- 묶는 열 + 집계
FROM     studies
WHERE    (행 조건)
GROUP BY 묶는열                        -- 결과 1행 = 그룹 1개
ORDER BY n DESC
```
**결과의 한 행이 무엇인지** 항상 말로 확인하세요: "한 행 = 임상 단계 하나".

In [ ]:
%%sql
-- 임상 단계별 시험 수  (한 행 = 단계 하나)
SELECT phase, COUNT(*) AS n
FROM studies
GROUP BY phase
ORDER BY n DESC

In [ ]:
%%sql
-- 연구 유형별 시험 수와 등록 인원 통계
SELECT study_type, COUNT(*) AS n, ROUND(AVG(enrollment)) AS 평균등록, MEDIAN(enrollment) AS 중앙값
FROM studies
GROUP BY study_type

In [ ]:
%%sql
-- 연도별 시작 시험 수  (YEAR() 로 날짜에서 연도만 뽑기)
SELECT YEAR(start_date) AS 시작연도, COUNT(*) AS n
FROM studies
GROUP BY 시작연도
ORDER BY 시작연도

In [ ]:
%%sql
-- 진행 상태별 분포(%)  — 마지막 열은 전체 대비 비율 (참고용)
SELECT overall_status, COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS 퍼센트
FROM studies
GROUP BY overall_status
ORDER BY n DESC

### 빈 값에 이름 붙이기 — COALESCE

`phase` 에는 **빈 값(NULL)** 과 글자 `'NA'` 가 **따로** 있습니다. 둘은 다른 값입니다.
빈 값은 그대로 두면 이름 없는 칸으로 나오니, `COALESCE` 로 이름을 붙여 드러냅니다.

In [ ]:
%%sql
-- 빈 값에 '미기재' 라는 이름을 붙여 함께 세기
SELECT COALESCE(phase, '미기재') AS 단계, COUNT(*) AS n
FROM studies
GROUP BY 단계
ORDER BY n DESC

In [ ]:
%%sql
-- 빠진 값이 몇 건인지 먼저 확인하는 습관
SELECT COUNT(*) AS 전체, COUNT(phase) AS 값있음, COUNT(*) - COUNT(phase) AS 빈값
FROM studies

### 구간으로 나누어 세기 — CASE WHEN

엑셀의 `IF` 와 같습니다. 위에서부터 차례로 맞는 조건을 찾고, 맞으면 거기서 멈춥니다.
어디에도 안 맞으면 `ELSE` 의 값이 됩니다.

```
CASE WHEN 조건1 THEN 값1
     WHEN 조건2 THEN 값2
     ELSE 값3
END AS 새이름
```

In [ ]:
%%sql
-- 등록 인원을 세 구간으로 나누어 세기
SELECT CASE WHEN enrollment < 50 THEN '소규모'
            WHEN enrollment < 500 THEN '중규모'
            ELSE '대규모'
       END AS 규모,
       COUNT(*) AS n
FROM studies
WHERE enrollment IS NOT NULL
GROUP BY 규모
ORDER BY n DESC

In [ ]:
%%sql
-- 분모와 분자를 한 번에 세기 — 병원의 지표 충족률과 같은 모양
SELECT overall_status AS 상태,
       COUNT(*) AS 전체,
       COUNT(CASE WHEN results_first_posted_date IS NOT NULL THEN 1 END) AS 결과게시,
       ROUND(100.0 * COUNT(CASE WHEN results_first_posted_date IS NOT NULL THEN 1 END) / COUNT(*), 1) AS 게시율
FROM studies
GROUP BY 상태
ORDER BY 전체 DESC
LIMIT 5

## 3. HAVING — 집계 결과에 조건

- `WHERE` 는 **묶기 전** 행에, `HAVING` 은 **묶은 뒤** 그룹에 조건을 겁니다.
- 순서: `SELECT → FROM → WHERE → GROUP BY → HAVING → ORDER BY → LIMIT`

In [ ]:
%%sql
-- 시험이 1000건 이상인 국가만  (WHERE: 제외된 국가 행 빼기 → GROUP BY → HAVING: 1000건 이상 그룹만)
SELECT name AS 국가, COUNT(*) AS n
FROM countries
WHERE removed = false
GROUP BY name
HAVING COUNT(*) >= 1000
ORDER BY n DESC

### 작성 순서 ≠ 실행 순서
| 작성 순서 | 실행 순서 | 하는 일 |
|---|---|---|
| SELECT | ⑤ | 보여줄 열·집계 계산 |
| FROM | ① | 테이블 가져오기 |
| WHERE | ② | 행 걸러내기 |
| GROUP BY | ③ | 묶기 |
| HAVING | ④ | 그룹 걸러내기 |
| ORDER BY | ⑥ | 정렬 |
| LIMIT | ⑦ | 앞부분만 |

그래서 `WHERE` 에는 집계 함수를 쓸 수 없습니다(아직 묶기 전). 집계에 대한 조건은 `HAVING` 에 씁니다.

## 4. JOIN — 두 테이블 연결하기

`studies` 에는 국가·질환 정보가 없습니다. 국가는 `countries`, 질환은 `browse_conditions` 에 **시험 1건당 여러 행**으로 따로 있습니다. 공통 열 `nct_id` 로 연결합니다.

```
SELECT s.nct_id, s.brief_title, c.name
FROM   studies AS s
JOIN   countries AS c ON s.nct_id = c.nct_id     -- 연결 키
```
- `JOIN`(= INNER JOIN): **양쪽에 다 있는** 행만
- `LEFT JOIN`: 왼쪽(studies)은 모두 남기고, 오른쪽에 없으면 NULL

In [ ]:
%%sql
-- 한국에서 진행된 시험 (studies ↔ countries)
SELECT s.nct_id, s.brief_title, c.name AS 국가
FROM studies AS s
JOIN countries AS c ON s.nct_id = c.nct_id
WHERE c.name = 'South Korea' AND c.removed = false
ORDER BY s.nct_id
LIMIT 10

In [ ]:
%%sql
-- 국가별 중재 시험 수와 등록 인원 중앙값 (500건 이상 국가)
SELECT c.name AS 국가, COUNT(DISTINCT s.nct_id) AS 시험수, MEDIAN(s.enrollment) AS 등록인원_중앙값
FROM studies AS s
JOIN countries AS c ON s.nct_id = c.nct_id
WHERE c.removed = false AND s.study_type = 'INTERVENTIONAL'
GROUP BY c.name
HAVING COUNT(DISTINCT s.nct_id) >= 500
ORDER BY 시험수 DESC

In [ ]:
%%sql
-- 질환(MeSH 표준 용어)별 시험 수 (studies ↔ browse_conditions)
SELECT b.mesh_term AS 질환, COUNT(DISTINCT s.nct_id) AS 시험수
FROM studies AS s
JOIN browse_conditions AS b ON s.nct_id = b.nct_id
WHERE b.mesh_term IN ('Diabetes Mellitus, Type 2', 'Hypertension', 'Heart Failure', 'Stroke', 'Asthma')
GROUP BY b.mesh_term
ORDER BY 시험수 DESC

In [ ]:
%%sql
-- 주관 스폰서 유형별 시험 수 (INDUSTRY = 제약사 등)
SELECT sp.agency_class AS 스폰서유형, COUNT(*) AS 시험수, MEDIAN(s.enrollment) AS 등록인원_중앙값
FROM studies AS s
JOIN sponsors AS sp ON s.nct_id = sp.nct_id
WHERE sp.lead_or_collaborator = 'lead'
GROUP BY sp.agency_class
ORDER BY 시험수 DESC

### JOIN 뒤에는 행 수를 꼭 확인 — 한 건이 여러 행으로 늘어납니다
한 시험에 질환(MeSH)이 평균 10개 붙어 있습니다. `browse_conditions` 를 붙인 뒤 `COUNT(*)` 를 세면 **시험 수가 아니라 (시험 × 질환) 쌍의 수**가 나옵니다.

In [ ]:
%%sql
-- JOIN 뒤 행 수 vs 시험 수
SELECT COUNT(*) AS 행수, COUNT(DISTINCT s.nct_id) AS 시험수
FROM studies AS s
JOIN browse_conditions AS b ON s.nct_id = b.nct_id

In [ ]:
%%sql
-- LEFT JOIN: 기관 정보가 없는 시험도 남김 (f.nct_id 가 NULL 인 행)
SELECT COUNT(*) AS 행수, COUNT(DISTINCT s.nct_id) AS 시험수, COUNT(f.nct_id) AS 기관있는행
FROM studies AS s
LEFT JOIN facilities AS f ON s.nct_id = f.nct_id

In [ ]:
%%sql
-- 기관 정보가 하나도 없는 시험 수  (LEFT JOIN + IS NULL)
SELECT COUNT(*) AS 기관정보없는시험
FROM studies AS s
LEFT JOIN facilities AS f ON s.nct_id = f.nct_id
WHERE f.nct_id IS NULL

## 5. 자주 하는 실수

오류가 나는 실수와 **오류 없이 틀린 답이 나오는 실수**를 구분해 보세요.

### 5-1. GROUP BY 에 없는 열을 SELECT 에 적음 (오류)

In [ ]:
%%sql
-- [오류 예시] overall_status 는 묶는 열도, 집계도 아님
SELECT phase, overall_status, COUNT(*) AS n
FROM studies
GROUP BY phase

In [ ]:
%%sql
-- 고친 버전: 두 열로 묶기
SELECT phase, overall_status, COUNT(*) AS n
FROM studies
GROUP BY phase, overall_status
ORDER BY n DESC
LIMIT 10

### 5-2. WHERE 에 집계 함수 (오류)

In [ ]:
%%sql
-- [오류 예시] WHERE 는 묶기 전이라 COUNT 를 모름
SELECT phase, COUNT(*) AS n
FROM studies
WHERE COUNT(*) > 1000
GROUP BY phase

In [ ]:
%%sql
-- 고친 버전: HAVING
SELECT phase, COUNT(*) AS n
FROM studies
GROUP BY phase
HAVING COUNT(*) > 1000
ORDER BY n DESC

### 5-3. JOIN 뒤 COUNT(*) 를 시험 수로 착각 (오류 없음)

In [ ]:
%%sql
-- [오류 예시] 한국 기관 수만큼 행이 늘어남 → 시험 수가 아님
SELECT COUNT(*) AS 한국시험수
FROM studies AS s
JOIN facilities AS f ON s.nct_id = f.nct_id
WHERE f.country = 'South Korea'

In [ ]:
%%sql
-- 고친 버전: COUNT(DISTINCT nct_id)
SELECT COUNT(DISTINCT s.nct_id) AS 한국시험수
FROM studies AS s
JOIN facilities AS f ON s.nct_id = f.nct_id
WHERE f.country = 'South Korea'

### 5-4. 평균만 보고 판단 (오류 없음)

In [ ]:
%%sql
-- [오류 예시] 관찰연구 평균 등록 인원 → 수만 명? 극단값 때문
SELECT ROUND(AVG(enrollment)) AS 평균등록인원
FROM studies
WHERE study_type = 'OBSERVATIONAL'

In [ ]:
%%sql
-- 고친 버전: 중앙값과 최대값을 함께 보기
SELECT ROUND(AVG(enrollment)) AS 평균, MEDIAN(enrollment) AS 중앙값, MAX(enrollment) AS 최대
FROM studies
WHERE study_type = 'OBSERVATIONAL'

### 5-5. 제외된 국가 행을 빼지 않음 (오류 없음)

In [ ]:
%%sql
-- [오류 예시] removed = true 인 행(등록 후 제외된 국가)까지 셈
SELECT COUNT(DISTINCT nct_id) AS n
FROM countries
WHERE name = 'United States'

In [ ]:
%%sql
-- 고친 버전: removed = false 조건 추가  (데이터 사전에서 removed 의 뜻을 확인)
SELECT COUNT(DISTINCT nct_id) AS n
FROM countries
WHERE name = 'United States' AND removed = false

### 절 순서 맞추기

아래 조각을 **올바른 순서로 다시 배열**해서 실행해 보세요.
순서를 바꿔 적으면 결과가 이상해지는 것이 아니라 **문법 오류**가 납니다.

```
적는 순서:  SELECT → FROM → WHERE → GROUP BY → HAVING → ORDER BY → LIMIT
```

**O1.** 조각: `LIMIT 5` / `GROUP BY phase` / `SELECT phase, COUNT(*) AS 건수` / `ORDER BY 건수 DESC` / `FROM studies` / `WHERE start_date >= '2024-01-01'`

2024년에 시작한 시험을 임상 단계별로 세고, 많은 순으로 위에서 5줄만 보세요.

In [ ]:
%%sql
-- O1
-- 아래 여섯 줄을 올바른 순서로 다시 적으세요
-- LIMIT 5
-- GROUP BY phase
-- SELECT phase, COUNT(*) AS 건수
-- ORDER BY 건수 DESC
-- FROM studies
-- WHERE start_date >= '2024-01-01'
___

In [ ]:
check("O1", _)

**O2.** 조각: `HAVING COUNT(*) >= 1000` / `FROM studies` / `ORDER BY 건수 DESC` / `SELECT overall_status, COUNT(*) AS 건수` / `GROUP BY overall_status`

진행 상태별 시험 수가 1000건 이상인 상태만 많은 순으로 보세요.

In [ ]:
%%sql
-- O2
-- 아래 다섯 줄을 올바른 순서로 다시 적으세요
-- HAVING COUNT(*) >= 1000
-- FROM studies
-- ORDER BY 건수 DESC
-- SELECT overall_status, COUNT(*) AS 건수
-- GROUP BY overall_status
___

In [ ]:
check("O2", _)

## 6. 직접 해보기

`___` 를 채우고 실행한 뒤, 다음 셀의 `check(...)` 로 확인하세요. 결과 열의 이름은 문항에 적힌 대로 `AS` 로 맞춰 주세요.

**Q1.** 연구 유형(`study_type`)별 시험 수를 `n` 으로 세고 많은 순으로 정렬하세요.

In [ ]:
%%sql
-- Q1
SELECT study_type, COUNT(*) AS n
FROM studies
GROUP BY ___
ORDER BY ___

In [ ]:
check("Q1", _)

**Q2.** 완료(COMPLETED)된 중재(INTERVENTIONAL) 시험의 등록 인원 평균(`평균`, 반올림)과 중앙값(`중앙값`)을 구하세요.

In [ ]:
%%sql
-- Q2
SELECT ROUND(AVG(enrollment)) AS 평균, MEDIAN(enrollment) AS 중앙값
FROM studies
WHERE ___

In [ ]:
check("Q2", _)

**Q3.** 시작 연도(`시작연도`)별로 PHASE3 시험 수(`n`)를 구하고 연도 순으로 정렬하세요.

In [ ]:
%%sql
-- Q3
SELECT YEAR(start_date) AS 시작연도, COUNT(*) AS n
FROM studies
WHERE ___
GROUP BY ___
ORDER BY ___

In [ ]:
check("Q3", _)

**Q4.** 임상 단계(`phase`)별 시험 수(`n`)가 1500건 이상인 단계만 많은 순으로 조회하세요. (HAVING)

In [ ]:
%%sql
-- Q4
SELECT phase, COUNT(*) AS n
FROM studies
GROUP BY phase
HAVING ___
ORDER BY n DESC

In [ ]:
check("Q4", _)

**Q5.** 국가(`국가`)별 시험 수(`시험수`)를 구하되 제외된 국가 행(`removed`)은 빼고, 2000건 이상인 국가만 많은 순으로 조회하세요.

In [ ]:
%%sql
-- Q5
SELECT name AS 국가, COUNT(DISTINCT nct_id) AS 시험수
FROM countries
WHERE ___
GROUP BY name
HAVING ___
ORDER BY 시험수 DESC

In [ ]:
check("Q5", _)

**Q6.** 한국(South Korea)에서 진행된 **PHASE3** 시험 수(`n`)를 구하세요. studies 와 countries 를 연결하고, 제외된 국가 행은 빼고, 시험을 중복 없이 세세요.

In [ ]:
%%sql
-- Q6
SELECT COUNT(DISTINCT s.nct_id) AS n
FROM studies AS s
JOIN countries AS c ON ___
WHERE c.name = ___ AND c.removed = false AND s.phase = ___

In [ ]:
check("Q6", _)

**Q7.** 중재 유형(`intervention_type`)별 시험 수(`시험수`, 중복 없이)를 많은 순으로 조회하세요. (studies ↔ interventions)

In [ ]:
%%sql
-- Q7
SELECT i.intervention_type, COUNT(DISTINCT s.nct_id) AS 시험수
FROM studies AS s
JOIN interventions AS i ON ___
GROUP BY ___
ORDER BY 시험수 DESC

In [ ]:
check("Q7", _)

**Q8.** 질환 'Heart Failure' 시험의 스폰서 유형(`agency_class`)별 시험 수(`시험수`, 중복 없이)를 많은 순으로 조회하세요. (browse_conditions, sponsors 중 lead 만)

In [ ]:
%%sql
-- Q8
SELECT sp.agency_class, COUNT(DISTINCT s.nct_id) AS 시험수
FROM studies AS s
JOIN browse_conditions AS b ON s.nct_id = b.nct_id
JOIN sponsors AS sp ON s.nct_id = sp.nct_id
WHERE b.mesh_term = ___ AND sp.lead_or_collaborator = ___
GROUP BY sp.agency_class
ORDER BY 시험수 DESC

In [ ]:
check("Q8", _)

## 7. 오늘 배운 것

- 집계는 **여러 행 → 하나의 값**. `COUNT(*)` 와 `COUNT(열)`, `COUNT(DISTINCT 열)` 은 서로 다릅니다.
- 치우친 값은 **평균과 중앙값을 함께** 봅니다.
- `GROUP BY` 결과의 한 행이 무엇인지 말로 확인합니다. 집계 조건은 `HAVING`.
- 작성 순서 `SELECT → FROM → WHERE → GROUP BY → HAVING → ORDER BY → LIMIT`, 실행은 `FROM` 부터.
- `JOIN` 뒤에는 행 수가 늘어날 수 있으므로 `COUNT(*)` 와 `COUNT(DISTINCT nct_id)` 를 같이 확인합니다.

다음 시간(9주차): 분석 질문을 단계별 SQL 로 바꾸는 종합 실습, 과제 1 제출.